# 01 — Design Patterns pythoniques

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- comprendre pourquoi les patterns classiques sont souvent simplifiés en Python
- implémenter le pattern Registry avec un décorateur
- implémenter le pattern Factory (simple et avec Protocol)
- comprendre le Singleton en Python (module, `__new__`, Borg)
- implémenter le pattern Strategy avec des fonctions ou Protocol
- implémenter le pattern Observer avec callbacks

## Prérequis — ce que vous connaissez déjà

À ce stade de la formation intermédiaire, vous maîtrisez :

- le modèle objet complet (classes, héritage, ABC, Protocol)
- les décorateurs simples et paramétrés, `functools.wraps`
- les context managers (`with`, `@contextmanager`)
- le typage moderne (Protocol, Generic, TypeVar)
- les dataclasses

Ce que nous n'avons **pas encore vu** (et que nous n'utiliserons donc pas dans ce notebook) :

- les metaclasses (pattern avancé hors programme intermédiaire)
- asyncio (observer async)

## Plan

1. Pourquoi les patterns sont différents en Python
2. Registry : enregistrer des plugins avec un décorateur
3. Factory : créer des objets sans coupler le code client
4. Singleton : instance unique
5. Strategy : algorithme interchangeable
6. Observer : notification d'événements
7. Synthèse
8. Exercices

---

## 1. Pourquoi les patterns sont différents en Python

Les design patterns du GoF (Gang of Four) viennent de Java/C++. En Python, beaucoup sont simplifiés grâce à :

- les **fonctions de première classe** (pas besoin de classes Strategy vides)
- les **décorateurs** (Registry en 5 lignes)
- les **modules** (un module est déjà un Singleton)
- le **duck typing** et les **Protocols** (pas besoin d'interfaces abstraites)
- les **dictionnaires** (dispatch table = switch/case dynamique)

Notre approche : pour chaque pattern, on montre d'abord la version **classique** (orientée classes), puis la version **pythonique** (simplifiée).

---

## 2. Registry : enregistrer des plugins avec un décorateur

Le Registry maintient un dictionnaire de composants enregistrés, accessibles par un nom ou un type.

### Version classique (Java-like)

In [ ]:
class Registry:
    _plugins: dict[str, type] = {}

    @classmethod
    def register(cls, name: str, plugin: type) -> None:
        cls._plugins[name] = plugin

    @classmethod
    def get(cls, name: str) -> type:
        return cls._plugins[name]


In [ ]:
class PluginCSV:
    def charger(self, path: str) -> str:
        return f"Chargement CSV de {path}"

Registry.register("csv", PluginCSV)
Registry.get("csv")().charger("data.csv")


### Version pythonique : décorateur + dictionnaire

In [ ]:
EXPORTERS: dict[str, type] = {}

def exporter(format_name: str):
    """Décorateur qui enregistre une classe comme exporteur."""
    def decorator(cls):
        EXPORTERS[format_name] = cls
        return cls
    return decorator


In [ ]:
@exporter("csv")
class CSVExporter:
    def export(self, data: list[str]) -> str:
        return ",".join(data)

@exporter("json")
class JSONExporter:
    def export(self, data: list[str]) -> str:
        import json
        return json.dumps(data)


In [ ]:
EXPORTERS


In [ ]:
# Utilisation
exp = EXPORTERS["csv"]()
exp.export(["Alice", "Bob", "Charlie"])


**Avantage :** pour ajouter un nouveau format, il suffit de décorer une nouvelle classe. Aucune modification du code existant (Open/Closed Principle).

### Variante : registre de fonctions

In [ ]:
HANDLERS: dict[str, callable] = {}

def handler(event: str):
    def decorator(func):
        HANDLERS[event] = func
        return func
    return decorator

@handler("click")
def on_click(x: int, y: int) -> str:
    return f"Click at ({x}, {y})"

@handler("keypress")
def on_key(key: str) -> str:
    return f"Key: {key}"

# Dispatch
HANDLERS["click"](100, 200)


---

## 3. Factory : créer des objets sans coupler le code client

Le Factory pattern encapsule la logique de création d'objets. Le code client ne connaît pas les classes concrètes.

### Simple Factory : une fonction

In [ ]:
from dataclasses import dataclass

@dataclass
class Cercle:
    rayon: float
    def aire(self) -> float:
        import math
        return math.pi * self.rayon ** 2

@dataclass
class Rectangle:
    largeur: float
    hauteur: float
    def aire(self) -> float:
        return self.largeur * self.hauteur


In [ ]:
def creer_forme(type_: str, **kwargs) -> Cercle | Rectangle:
    """Factory function."""
    formes = {"cercle": Cercle, "rectangle": Rectangle}
    if type_ not in formes:
        raise ValueError(f"Forme inconnue : {type_}")
    return formes[type_](**kwargs)


In [ ]:
c = creer_forme("cercle", rayon=5)
r = creer_forme("rectangle", largeur=3, hauteur=4)
print(f"Cercle : {c.aire():.2f}")
print(f"Rectangle : {r.aire():.2f}")


### Factory + Registry : le combo

In [ ]:
from typing import Protocol

class Forme(Protocol):
    def aire(self) -> float: ...

FORMES: dict[str, type] = {}

def forme(name: str):
    def deco(cls):
        FORMES[name] = cls
        return cls
    return deco

def fabriquer(name: str, **kwargs) -> Forme:
    return FORMES[name](**kwargs)


In [ ]:
@forme("triangle")
@dataclass
class Triangle:
    base: float
    hauteur: float
    def aire(self) -> float:
        return self.base * self.hauteur / 2

fabriquer("triangle", base=6, hauteur=3).aire()


---

## 4. Singleton : instance unique

Le Singleton garantit qu'une classe n'a qu'une seule instance. En Python, il y a **trois approches** courantes.

### Approche 1 : le module (la plus pythonique)

Un module Python est déjà un singleton : il n'est importé qu'une fois. Si vous n'avez besoin que de **données + fonctions**, un module suffit.

```python
# config.py
settings = {"debug": False, "db_url": "sqlite:///app.db"}

def get_setting(key: str) -> str:
    return settings[key]
```

### Approche 2 : `__new__`

In [ ]:
class Singleton:
    _instance: "Singleton | None" = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self) -> None:
        self.value = 42


In [ ]:
a = Singleton()
b = Singleton()
print(a is b)     # True
print(id(a) == id(b))  # True


### Approche 3 : le pattern Borg (shared state)

In [ ]:
class Borg:
    _shared_state: dict = {}

    def __init__(self) -> None:
        self.__dict__ = self._shared_state

a = Borg()
a.x = 10
b = Borg()
print(b.x)        # 10
print(a is b)      # False — instances différentes
print(a.x is b.x)  # True — mais même état


**Recommandation :** préférez le module-singleton. N'utilisez `__new__` ou Borg que si vous avez une vraie contrainte d'interface.

---

## 5. Strategy : algorithme interchangeable

Le pattern Strategy permet de changer d'algorithme à l'exécution. En Python, une **fonction** est déjà une stratégie.

### Version classique : avec des classes

In [ ]:
from typing import Protocol

class DiscountStrategy(Protocol):
    def calculer(self, prix: float) -> float: ...

class SansRemise:
    def calculer(self, prix: float) -> float:
        return prix

class Remise10:
    def calculer(self, prix: float) -> float:
        return prix * 0.9

class RemiseFidelite:
    def calculer(self, prix: float) -> float:
        return prix * 0.85


In [ ]:
def facturer(prix: float, strategie: DiscountStrategy) -> float:
    return strategie.calculer(prix)

print(facturer(100, SansRemise()))
print(facturer(100, Remise10()))
print(facturer(100, RemiseFidelite()))


### Version pythonique : des fonctions

In [ ]:
from typing import Callable

def sans_remise(prix: float) -> float:
    return prix

def remise_10(prix: float) -> float:
    return prix * 0.9

def remise_fidelite(prix: float) -> float:
    return prix * 0.85


In [ ]:
def facturer_v2(prix: float, strategie: Callable[[float], float]) -> float:
    return strategie(prix)

print(facturer_v2(100, sans_remise))
print(facturer_v2(100, remise_10))
print(facturer_v2(100, remise_fidelite))


Beaucoup plus simple ! En Python, si la stratégie est une seule méthode, une **fonction** suffit. Si elle a de l'état ou plusieurs méthodes, une classe reste pertinente.

### Strategy avec des lambdas ou `functools.partial`

In [ ]:
from functools import partial

def remise_pourcentage(prix: float, pourcent: float) -> float:
    return prix * (1 - pourcent / 100)

remise_15 = partial(remise_pourcentage, pourcent=15)
facturer_v2(100, remise_15)


---

## 6. Observer : notification d'événements

L'Observer (ou Pub/Sub) permet à un objet de notifier ses abonnés quand son état change.

### Version minimale avec des callbacks

In [ ]:
from typing import Callable

class EventEmitter:
    def __init__(self) -> None:
        self._listeners: dict[str, list[Callable]] = {}

    def on(self, event: str, callback: Callable) -> None:
        self._listeners.setdefault(event, []).append(callback)

    def emit(self, event: str, **kwargs) -> None:
        for cb in self._listeners.get(event, []):
            cb(**kwargs)


In [ ]:
bus = EventEmitter()

def on_user_created(name: str) -> None:
    print(f"Bienvenue {name} !")

def on_user_created_log(name: str) -> None:
    print(f"[LOG] Utilisateur créé : {name}")

bus.on("user_created", on_user_created)
bus.on("user_created", on_user_created_log)

bus.emit("user_created", name="Alice")


### Version avec décorateur d'abonnement

In [ ]:
class EventBus:
    def __init__(self) -> None:
        self._listeners: dict[str, list[Callable]] = {}

    def subscribe(self, event: str):
        """Décorateur pour abonner une fonction à un événement."""
        def decorator(func):
            self._listeners.setdefault(event, []).append(func)
            return func
        return decorator

    def emit(self, event: str, **kwargs) -> None:
        for cb in self._listeners.get(event, []):
            cb(**kwargs)


In [ ]:
bus2 = EventBus()

@bus2.subscribe("order_placed")
def send_email(order_id: int) -> None:
    print(f"Email envoyé pour commande #{order_id}")

@bus2.subscribe("order_placed")
def update_stock(order_id: int) -> None:
    print(f"Stock mis à jour pour commande #{order_id}")

bus2.emit("order_placed", order_id=42)


### Observer typé avec Protocol

In [ ]:
from typing import Protocol

class OrderObserver(Protocol):
    def on_order(self, order_id: int, total: float) -> None: ...

class EmailNotifier:
    def on_order(self, order_id: int, total: float) -> None:
        print(f"Email: commande #{order_id}, total={total:.2f}")

class StockManager:
    def on_order(self, order_id: int, total: float) -> None:
        print(f"Stock: mise à jour pour #{order_id}")

class OrderService:
    def __init__(self) -> None:
        self.observers: list[OrderObserver] = []

    def add_observer(self, obs: OrderObserver) -> None:
        self.observers.append(obs)

    def place_order(self, order_id: int, total: float) -> None:
        print(f"Commande #{order_id} passée")
        for obs in self.observers:
            obs.on_order(order_id, total)

service = OrderService()
service.add_observer(EmailNotifier())
service.add_observer(StockManager())
service.place_order(1, 99.90)


---

## Synthèse

| Pattern | Version classique | Version pythonique |
|---------|------------------|--------------------|
| Registry | Classe avec `register()` | Décorateur + dict |
| Factory | Classe abstraite + sous-classes | Fonction + dict de constructeurs |
| Singleton | `__new__` | Module (le plus souvent) |
| Strategy | Interface + classes concrètes | Fonctions / `Callable` |
| Observer | Interface Observer | Callbacks + EventEmitter |

### Règles à retenir

1. Toujours préférer la version **pythonique** (simple) en premier.
2. Le **décorateur + dict** remplace 90 % des Registry/Factory classiques.
3. Le **module** est le Singleton naturel de Python.
4. Une **fonction** est la Strategy la plus simple.
5. L'Observer avec **callbacks** est plus flexible que l'héritage.
6. Utiliser `Protocol` pour typer les interfaces quand le duck typing ne suffit pas.

---

## Exercices

Les exercices sont gradués. Tous utilisent des fonctions typées (PEP 604).

### Exercice 1 — Registre de sérialiseurs *(facile)*

Créez un registre `SERIALIZERS` avec un décorateur `@serializer(format_name)`. Implémentez un sérialiseur JSON et un sérialiseur YAML (simulé avec `repr`).

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Patterns_pythoniques", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
import json

SERIALIZERS: dict[str, type] = {}

def serializer(name: str):
    def deco(cls):
        SERIALIZERS[name] = cls
        return cls
    return deco

@serializer("json")
class JSONSerializer:
    def serialize(self, data: dict) -> str:
        return json.dumps(data)

@serializer("yaml")
class YAMLSerializer:
    def serialize(self, data: dict) -> str:
        return repr(data)  # simulé

s = SERIALIZERS["json"]()
print(s.serialize({"a": 1}))
```

</details>

### Exercice 2 — Strategy de tri *(facile)*

Écrivez 3 fonctions de tri (par nom, par age, par score) et une fonction `trier_utilisateurs(users, strategy)` qui accepte une `Callable` comme stratégie.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Patterns_pythoniques", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
from typing import Callable

users = [
    {"nom": "Charlie", "age": 25, "score": 88},
    {"nom": "Alice", "age": 30, "score": 95},
    {"nom": "Bob", "age": 22, "score": 72},
]

def par_nom(u: dict) -> str:
    return u["nom"]

def par_age(u: dict) -> int:
    return u["age"]

def par_score(u: dict) -> int:
    return u["score"]

def trier_utilisateurs(users: list[dict], strategy: Callable) -> list[dict]:
    return sorted(users, key=strategy)

for s in [par_nom, par_age, par_score]:
    print(f"{s.__name__}: {trier_utilisateurs(users, s)}")
```

</details>

### Exercice 3 — EventBus avec unsubscribe *(moyen)*

Étendez la classe `EventBus` pour ajouter une méthode `unsubscribe(event, func)` qui retire un callback d'un événement.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Patterns_pythoniques", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
from typing import Callable

class EventBus:
    def __init__(self) -> None:
        self._listeners: dict[str, list[Callable]] = {}

    def subscribe(self, event: str):
        def decorator(func):
            self._listeners.setdefault(event, []).append(func)
            return func
        return decorator

    def unsubscribe(self, event: str, func: Callable) -> None:
        if event in self._listeners:
            self._listeners[event].remove(func)

    def emit(self, event: str, **kwargs) -> None:
        for cb in self._listeners.get(event, []):
            cb(**kwargs)

bus = EventBus()

@bus.subscribe("test")
def handler(msg: str) -> None:
    print(f"Handler: {msg}")

bus.emit("test", msg="hello")
bus.unsubscribe("test", handler)
bus.emit("test", msg="silent")  # rien
```

</details>

### Exercice 4 — Factory avec validation *(moyen)*

Créez une factory `creer_vehicule(type_, **kwargs)` qui :

- supporte 'voiture', 'moto', 'velo'
- chaque véhicule a des attributs différents
- lève `ValueError` si un attribut obligatoire manque

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Patterns_pythoniques", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
from dataclasses import dataclass

@dataclass
class Voiture:
    marque: str
    nb_portes: int

@dataclass
class Moto:
    marque: str
    cylindree: int

@dataclass
class Velo:
    type_: str  # route, VTT, ville

VEHICULES = {
    "voiture": Voiture,
    "moto": Moto,
    "velo": Velo,
}

def creer_vehicule(type_: str, **kwargs):
    if type_ not in VEHICULES:
        raise ValueError(f"Type inconnu : {type_}")
    try:
        return VEHICULES[type_](**kwargs)
    except TypeError as e:
        raise ValueError(f"Attributs invalides : {e}") from e

print(creer_vehicule("voiture", marque="Peugeot", nb_portes=5))
print(creer_vehicule("moto", marque="Yamaha", cylindree=600))
```

</details>

### Exercice 5 — Pipeline de transformations (Chain of Responsibility) *(difficile)*

Créez un `Pipeline` qui chaîne des fonctions de transformation de texte. Chaque étape est une `Callable[[str], str]` enregistrée via `add_step`. La méthode `execute(text)` exécute les étapes dans l'ordre.

Bonus : ajoutez un décorateur `@pipeline.step` pour enregistrer directement.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Patterns_pythoniques", exercice=5)


<details>
<summary>📖 Voir la correction</summary>

```python
from typing import Callable

class Pipeline:
    def __init__(self) -> None:
        self._steps: list[Callable[[str], str]] = []

    def add_step(self, func: Callable[[str], str]) -> None:
        self._steps.append(func)

    def step(self, func: Callable[[str], str]) -> Callable[[str], str]:
        """Décorateur."""
        self._steps.append(func)
        return func

    def execute(self, text: str) -> str:
        for step in self._steps:
            text = step(text)
        return text

p = Pipeline()

@p.step
def strip_ws(text: str) -> str:
    return text.strip()

@p.step
def lower(text: str) -> str:
    return text.lower()

@p.step
def remove_punct(text: str) -> str:
    return "".join(c for c in text if c.isalnum() or c.isspace())

print(p.execute("  Hello, World!  "))  # hello world
```

</details>

### Exercice 6 — Middleware HTTP (décorateur en chaîne) *(difficile)*

Simulez un système de middleware HTTP :

- Une requête est un `dict` avec `method`, `path`, `headers`
- Un middleware est un décorateur qui peut modifier la requête avant l'appel et la réponse après
- Implémentez `@auth_middleware` (vérifie un header) et `@logging_middleware`

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Patterns_pythoniques", exercice=6)


<details>
<summary>📖 Voir la correction</summary>

```python
from functools import wraps

def logging_middleware(func):
    @wraps(func)
    def wrapper(request: dict) -> dict:
        print(f"[LOG] {request['method']} {request['path']}")
        response = func(request)
        print(f"[LOG] Status: {response.get('status', 200)}")
        return response
    return wrapper

def auth_middleware(func):
    @wraps(func)
    def wrapper(request: dict) -> dict:
        if "Authorization" not in request.get("headers", {}):
            return {"status": 401, "body": "Non autorisé"}
        return func(request)
    return wrapper

@logging_middleware
@auth_middleware
def handle_request(request: dict) -> dict:
    return {"status": 200, "body": "OK"}

req_ok = {"method": "GET", "path": "/api", "headers": {"Authorization": "Bearer xxx"}}
print(handle_request(req_ok))

req_ko = {"method": "GET", "path": "/api", "headers": {}}
print(handle_request(req_ko))
```

</details>

---

## Ressources externes

### Documentation officielle
- [Design Patterns (refactoring.guru)](https://refactoring.guru/design-patterns/python)

### PEPs de référence
- [PEP 3107 — Function Annotations](https://peps.python.org/pep-3107/)
- [PEP 544 — Protocols: Structural subtyping](https://peps.python.org/pep-0544/)

### Lectures complémentaires
- Fluent Python, ch. 10 « Design Patterns with First-Class Functions »
- Python Patterns (brandon-rhodes.github.io/python-patterns)